# Part 1 (Complete, do not touch)

In [2]:
!pip install transformers evaluate accelerate datasets bert_score rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.5/232.5 kB 15.5 MB/

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import evaluate

In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
pd.options.display.max_colwidth = 200
perplexity = evaluate.load("perplexity", module_type="metric")

## Step 1

In [6]:
# Prompt entry
prompts = [
    'A serious and good philosophical work could',
    'The limits of my language means',
    'I don\'t know why we are here, but',
    'The real question of life after death',
    'A man will be imprisoned in a room',
]

In [7]:
# cleans a dictionary for output in pandas dataframe
def format_dict(dict): return '\n'.join(f'{k}: {v}' for k, v in dict.items())

In [8]:
# processes outputs for each algorithm for a given prompt and parameters
def generate_outputs(prompt, parameters):
  max_length = parameters['max_length']
  num_beams = parameters['num_beams']
  top_k = parameters['top_k']
  top_p = parameters['top_p']

  outputs = {}
  perplexities = {}

  input_ids = tokenizer(prompt, return_tensors="pt").input_ids

  # greedy search is the default behavior of model generation
  greedy_outputs = model.generate(input_ids, pad_token_id=tokenizer.eos_token_id, do_sample=False, max_length=max_length)
  # beam search is enabled whenever num_beams is specified above 1
  beam_outputs = model.generate(input_ids, pad_token_id=tokenizer.eos_token_id, do_sample=False, max_length=max_length, num_beams=num_beams)
  # do_sample enables sampling and disables greedy search
  # top_k specifies the number most likely tokens to maintain
  top_k_outputs = model.generate(input_ids, pad_token_id=tokenizer.eos_token_id, do_sample=True, max_length=max_length, top_k=top_k)
  # top_p specifies the cumulative probability threshold that is considered for token selection
  top_p_outputs = model.generate(input_ids, pad_token_id=tokenizer.eos_token_id, do_sample=True, max_length=max_length, top_p=top_p)

  outputs['prompt'] = prompt

  outputs['greedy'] = tokenizer.batch_decode(greedy_outputs, skip_special_tokens=True)
  outputs['beam'] = tokenizer.batch_decode(beam_outputs, skip_special_tokens=True)
  outputs['top k'] = tokenizer.batch_decode(top_k_outputs, skip_special_tokens=True)
  outputs['top p'] = tokenizer.batch_decode(top_p_outputs, skip_special_tokens=True)

  outputs['parameters'] = format_dict(parameters)

  # compute perplexities for each algorithm
  perplexities['greedy'] = perplexity.compute(model_id="gpt2", predictions=outputs['greedy'])['mean_perplexity']
  perplexities['beam'] = perplexity.compute(model_id="gpt2", predictions=outputs['beam'])['mean_perplexity']
  perplexities['top k'] = perplexity.compute(model_id="gpt2", predictions=outputs['top k'])['mean_perplexity']
  perplexities['top p'] = perplexity.compute(model_id="gpt2", predictions=outputs['top p'])['mean_perplexity']

  outputs['perplexity'] = format_dict(perplexities)

  return outputs

In [9]:
# driver function
def compare_algorithms(prompts):
  parameters = {
    'max_length': 30,
    'num_beams': 5,
    'top_k': 50,
    'top_p': 0.95
  }
  rows = []
  for prompt in prompts:
    row = generate_outputs(prompt, parameters)
    rows.append(row)

  results_df = pd.DataFrame(rows)
  return results_df

In [10]:
results = compare_algorithms(prompts)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
display(results)

,prompt,greedy,beam,top k,top p,parameters,perplexity
0,A serious and good philosophical work could,[A serious and good philosophical work could be done in the future.\n\nThe first step is to find a way to make the work more accessible to],[A serious and good philosophical work could not have been written without it.\n\nI am not a philosopher. I am not a scientist. I am],"[A serious and good philosophical work could be published in any number of ways. In the early 19th century, William Butler Yeats famously put forward a]","[A serious and good philosophical work could be done within the context of such a project.\n\nHowever, such a project would necessitate a substantial step]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 20.087020874023438\nbeam: 12.959490776062012\ntop k: 25.13960838317871\ntop p: 25.614749908447266
1,The limits of my language means,[The limits of my language means that I can't speak it. I can't write it. I can't write it. I can't write it],[The limits of my language means that I don't know what I'm talking about. I don't know what I'm talking about. I don't],[The limits of my language means a certain amount of power is necessary to express his meaning; and it is my duty to say thus about the words and],[The limits of my language means that if you're willing to put yourself in the middle of someone else's problem-solving process then it will be],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 6.806146144866943\nbeam: 6.083348751068115\ntop k: 37.699424743652344\ntop p: 19.26453399658203
2,"I don't know why we are here, but","[I don't know why we are here, but we are here to help you. We are here to help you. We are here to help you]","[I don't know why we are here, but we are here. We are here. We are here. We are here. We are here.]","[I don't know why we are here, but the game needs to change. It's a matter of time before the game comes to America.""\n]","[I don't know why we are here, but because of this tragedy we have to change our mentality about how to act.\n\n""I can]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 4.032037734985352\nbeam: 3.4798924922943115\ntop k: 12.607850074768066\ntop p: 13.887611389160156
3,The real question of life after death,"[The real question of life after death is not whether you will live or die, but whether you will live or die.\n\nThe question of life]","[The real question of life after death is what happens to the body after death, and what happens to the soul after death.\n\nThe question of]","[The real question of life after death is how many children, if any, are born into this world, at least by the time adults die?\n]","[The real question of life after death in this world, then, is whether any of the following might happen, for in all its mysteries of the world]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 6.118663311004639\nbeam: 6.55849027633667\ntop k: 15.893404006958008\ntop p: 32.61334228515625
4,A man will be imprisoned in a room,[A man will be imprisoned in a room with a woman for the rest of his life.\n\nThe man will be sentenced to life in prison.],[A man will be imprisoned in a room where he will not be able to speak.\n\nA man will be imprisoned in a room where he will],"[A man will be imprisoned in a room with a small, unmarked manhole. It is important that his release and sentence have the right message: Don]",[A man will be imprisoned in a room for the rest of his life after a hit-and-run crash involving a Porsche 535.\n\n],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 8.941352844238281\nbeam: 6.350132465362549\ntop k: 57.36750793457031\ntop p: 15.031204223632812


# Part 2

In [12]:
# I figure it'll be useful to do the dataset and model used in the example,
# then if we have issues there's a higher chance the prof/TAs will be familiar

#Loading dataset used in example
from datasets import load_dataset

ds = load_dataset("EdinburghNLP/xsum")

#Loading model used in example
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("gniemiec/t5-small-finetuned-xsum")
model = AutoModelForSeq2SeqLM.from_pretrained("gniemiec/t5-small-finetuned-xsum")

README.md:   0%|          | 0.00/6.24k [00:00<?, ?B/s]

xsum.py:   0%|          | 0.00/5.76k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/242M [00:00<?, ?B/s]

In [13]:
documents = ds["test"]["document"][0:50]
#stripping commas to see if this improves performance in any way
for i in range(len(documents)):
  documents[i] = documents[i].replace(',', '')
references = ds["test"]["summary"][0:50]

results = compare_algorithms(documents)
results["references"] = tokenizer.batch_decode(tokenizer(references, return_tensors="pt", padding=True).input_ids, skip_special_tokens=True) # Added padding which seemed to fix errors later on
display(results)

Token indices sequence length is longer than the specified maximum sequence length for this model (744 > 512). Running this sequence through the model will result in indexing errors


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

,prompt,greedy,beam,top k,top p,parameters,perplexity,references
0,Prison Link Cymru had 1099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in hou...,"[in Wales, a charity that has urged people to stay in a secure housing estate has said it is ""chronic"" for]","[in Wales, the Welsh Government has urged people to stay in a secure housing estate.]",[for the next five years and will run into it. who have been targeted by prison force members in Wales has been put in this post],"[to live in a secure apartment in Wales. It's a new, exciting and innovative development to use in housing in South Wales]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 46.846336364746094\nbeam: 33.5189094543457\ntop k: 148.06150817871094\ntop p: 58.72018051147461,"There is a ""chronic"" need for more housing for prison leavers in Wales, according to a charity."
1,Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.\nDetectives said three firearms ammunition and a five-figure sum of money were recovered.\nA 2...,[Police have recovered three firearms ammunition and a five-figure sum of money.],[Police have recovered three firearms ammunition and a five-figure sum of money.],"[are in custody and have arrested 26-year-old Amarsh Scott, whose family has been forced to return the missing person]",[Police Officers have recovered three firearms ammunition and five-figure sums of money.],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 41.20834732055664\nbeam: 41.20834732055664\ntop k: 68.57975006103516\ntop p: 87.23341369628906,"A man has appeared in court after firearms, ammunition and cash were seized by police in Edinburgh."
2,Jordan Hill Brittany Covington and Tesfaye Cooper all 18 and Tanishia Covington 24 appeared in a Chicago court on Friday.\nThe four have been charged with hate crimes and aggravated kidnapping and...,"[and,, and have been charged with hate crimes and aggravated kidnapping. and have been]","[who have been charged with hate crimes, aggravated kidnapping and battery among other things.]",[All-American. have been charged with hate crimes and aggravated kidnapping as three assailants are found],"[An Alabama woman who has been sentenced to 21 years in prison is awaiting trial for assaulting his mother, Tim Hill, in ]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 60.31381607055664\nbeam: 46.98090362548828\ntop k: 133.20643615722656\ntop p: 39.66783142089844,"Four people accused of kidnapping and torturing a mentally disabled man in a ""racially motivated"" attack streamed on Facebook have been denied bail."
3,The 48-year-old former Arsenal goalkeeper played for the Royals for four years.\nHe was appointed youth academy director in 2000 and has been director of football since 2003.\nA West Brom statemen...,[keeper David West Brom has been appointed a youth academy director of football since 2003.],['s former Arsenal goalkeeper has played for the Royals for four years and has been director of football since 2003.],[goalkeeper David Eldrado spent eight months working in the Athletic Union Division.],[Arsenal goalkeeper Ibrahim Ibrahimovic has been appointed manager of the University of Newcastle United.],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 209.69117736816406\nbeam: 33.44632339477539\ntop k: 398.7690124511719\ntop p: 36.21345901489258,"West Brom have appointed Nicky Hammond as technical director, ending his 20-year association with Reading."
4,Restoring the function of the organ - which helps control blood sugar levels - reversed symptoms of diabetes in animal experiments.\nThe study published in the journal Cell says the diet reboots t...,"[and. ""The results of the study have been very exciting,"" said Dr James Longo. ""The results of the study have been]","[. ""I was glad to have something to eat"" in a diet that 

In [14]:
import evaluate
rouge = evaluate.load("rouge")
greedy_rouge = []
beam_rouge = []
top_k_rouge = []
top_p_rouge = []

for i in range(50):
  greedy_rouge.append(rouge.compute(predictions=[results["greedy"][i][0]], references=[results["references"][i]]))
  beam_rouge.append(rouge.compute(predictions=[results["beam"][i][0]], references=[results["references"][i]]))
  top_k_rouge.append(rouge.compute(predictions=[results["top k"][i][0]], references=[results["references"][i]]))
  top_p_rouge.append(rouge.compute(predictions=[results["top p"][i][0]], references=[results["references"][i]]))

results["greedy rouge"] = greedy_rouge
results["beam rouge"] = beam_rouge
results["top k rouge"] = top_k_rouge
results["top p rouge"] = top_p_rouge
display(results)

,prompt,greedy,beam,top k,top p,parameters,perplexity,references,greedy rouge,beam rouge,top k rouge,top p rouge
0,Prison Link Cymru had 1099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in hou...,"[in Wales, a charity that has urged people to stay in a secure housing estate has said it is ""chronic"" for]","[in Wales, the Welsh Government has urged people to stay in a secure housing estate.]",[for the next five years and will run into it. who have been targeted by prison force members in Wales has been put in this post],"[to live in a secure apartment in Wales. It's a new, exciting and innovative development to use in housing in South Wales]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 46.846336364746094\nbeam: 33.5189094543457\ntop k: 148.06150817871094\ntop p: 58.72018051147461,"There is a ""chronic"" need for more housing for prison leavers in Wales, according to a charity.","{'rouge1': 0.5263157894736842, 'rouge2': 0.11111111111111112, 'rougeL': 0.21052631578947367, 'rougeLsum': 0.21052631578947367}","{'rouge1': 0.3125, 'rouge2': 0.06666666666666667, 'rougeL': 0.25, 'rougeLsum': 0.25}","{'rouge1': 0.186046511627907, 'rouge2': 0.048780487804878044, 'rougeL': 0.186046511627907, 'rougeLsum': 0.186046511627907}","{'rouge1': 0.3, 'rouge2': 0.052631578947368425, 'rougeL': 0.19999999999999998, 'rougeLsum': 0.19999999999999998}"
1,Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.\nDetectives said three firearms ammunition and a five-figure sum of money were recovered.\nA 2...,[Police have recovered three firearms ammunition and a five-figure sum of money.],[Police have recovered three firearms ammunition and a five-figure sum of money.],"[are in custody and have arrested 26-year-old Amarsh Scott, whose family has been forced to return the missing person]",[Police Officers have recovered three firearms ammunition and five-figure sums of money.],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 41.20834732055664\nbeam: 41.20834732055664\ntop k: 68.57975006103516\ntop p: 87.23341369628906,"A man has appeared in court after firearms, ammunition and cash were seized by police in Edinburgh.","{'rouge1': 0.33333333333333337, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}","{'rouge1': 0.33333333333333337, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}","{'rouge1': 0.15789473684210528, 'rouge2': 0.0, 'rougeL': 0.10526315789473684, 'rougeLsum': 0.10526315789473684}","{'rouge1': 0.26666666666666666, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}"
2,Jordan Hill Brittany Covington and Tesfaye Cooper all 18 and Tanishia Covington 24 appeared in a Chicago court on Friday.\nThe four have been charged with hate crimes and aggravated kidnapping and...,"[and,, and have been charged with hate crimes and aggravated kidnapping. and have been]","[who have been charged with hate crimes, aggravated kidnapping and battery among other things.]",[All-American. have been charged with hate crimes and aggravated kidnapping as three assailants are found],"[An Alabama woman who has been sentenced to 21 years in prison is awaiting trial for assaulting his mother, Tim Hill, in ]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 60.31381607055664\nbeam: 46.98090362548828\ntop k: 133.20643615722656\ntop p: 39.66783142089844,"Four people accused of kidnapping and torturing a mentally disabled man in a ""racially motivated"" attack streamed on Facebook have been denied bail.","{'rouge1': 0.2162162162162162, 'rouge2': 0.1142857142857143, 'rougeL': 0.2162162162162162, 'rougeLsum': 0.2162162162162162}","{'rouge1': 0.2162162162162162, 'rouge2': 0.1142857142857143, 'rougeL': 0.1081081081081081, 'rougeLsum': 0.1081081081081

In [16]:
import evaluate
bertscore = evaluate.load("bertscore")
greedy_bertscore = []
beam_bertscore = []
top_k_bertscore = []
top_p_bertscore = []

for i in range(50):
  greedy_bertscore.append(bertscore.compute(predictions=[results["greedy"][i][0]], references=[results["references"][i]], lang = 'en'))
  beam_bertscore.append(bertscore.compute(predictions=[results["beam"][i][0]], references=[results["references"][i]], lang = 'en'))
  top_k_bertscore.append(bertscore.compute(predictions=[results["top k"][i][0]], references=[results["references"][i]], lang = 'en'))
  top_p_bertscore.append(bertscore.compute(predictions=[results["top p"][i][0]], references=[results["references"][i]], lang = 'en'))

results["greedy bert"] = greedy_bertscore
results["beam bert"] = beam_bertscore
results["top k bert"] = top_k_bertscore
results["top p bert"] = top_p_bertscore
display(results)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,prompt,greedy,beam,top k,top p,parameters,perplexity,references,greedy rouge,beam rouge,top k rouge,top p rouge,greedy bert,beam bert,top k bert,top p bert
0,Prison Link Cymru had 1099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.\nWorkers at the charity claim investment in hou...,"[in Wales, a charity that has urged people to stay in a secure housing estate has said it is ""chronic"" for]","[in Wales, the Welsh Government has urged people to stay in a secure housing estate.]",[for the next five years and will run into it. who have been targeted by prison force members in Wales has been put in this post],"[to live in a secure apartment in Wales. It's a new, exciting and innovative development to use in housing in South Wales]",max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 46.846336364746094\nbeam: 33.5189094543457\ntop k: 148.06150817871094\ntop p: 58.72018051147461,"There is a ""chronic"" need for more housing for prison leavers in Wales, according to a charity.","{'rouge1': 0.5263157894736842, 'rouge2': 0.11111111111111112, 'rougeL': 0.21052631578947367, 'rougeLsum': 0.21052631578947367}","{'rouge1': 0.3125, 'rouge2': 0.06666666666666667, 'rougeL': 0.25, 'rougeLsum': 0.25}","{'rouge1': 0.186046511627907, 'rouge2': 0.048780487804878044, 'rougeL': 0.186046511627907, 'rougeLsum': 0.186046511627907}","{'rouge1': 0.3, 'rouge2': 0.052631578947368425, 'rougeL': 0.19999999999999998, 'rougeLsum': 0.19999999999999998}","{'precision': [0.8771550059318542], 'recall': [0.8916277289390564], 'f1': [0.8843321800231934], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}","{'precision': [0.8947465419769287], 'recall': [0.8781324625015259], 'f1': [0.8863616585731506], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}","{'precision': [0.8358310461044312], 'recall': [0.8665710687637329], 'f1': [0.8509235382080078], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}","{'precision': [0.8721945285797119], 'recall': [0.8638125658035278], 'f1': [0.867983341217041], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}"
1,Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.\nDetectives said three firearms ammunition and a five-figure sum of money were recovered.\nA 2...,[Police have recovered three firearms ammunition and a five-figure sum of money.],[Police have recovered three firearms ammunition and a five-figure sum of money.],"[are in custody and have arrested 26-year-old Amarsh Scott, whose family has been forced to return the missing person]",[Police Officers have recovered three firearms ammunition and five-figure sums of money.],max_length: 30\nnum_beams: 5\ntop_k: 50\ntop_p: 0.95,greedy: 41.20834732055664\nbeam: 41.20834732055664\ntop k: 68.57975006103516\ntop p: 87.23341369628906,"A man has appeared in court after firearms, ammunition and cash were seized by police in Edinburgh.","{'rouge1': 0.33333333333333337, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}","{'rouge1': 0.33333333333333337, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}","{'rouge1': 0.15789473684210528, 'rouge2': 0.0, 'rougeL': 0.10526315789473684, 'rougeLsum': 0.10526315789473684}","{'rouge1': 0.26666666666666666, 'rouge2': 0.14285714285714288, 'rougeL': 0.20000000000000004, 'rougeLsum': 0.20000000000000004}","{'precision': [0.8914594650268555], 'recall': [0.8753349781036377], 'f1': [0.8833236694335938], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}","{'precision': [0.8914594650268555], 'recall': [0.8753349781036377], 'f1': [0.8833236694335938], 'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0)'}","{'precision': [0.8504212498664856], 'recall': [0.8522648811340332], 'f1': [0.8513420820236206], 'hashcode': 